# Organised Tree Tracking (SIT)



Notebook-first tracking framework:

- Prints key textual diagnostics here.

- Saves all figures/artifacts under a single output folder.



## Inputs

- Orthomosaics: `input/input_om_sit/`

- Multi-threshold crowns: `output/input_crowns_multithreshold_min0p15/`



## Outputs

Choose `RUN_OUTPUT_DIR` below; the pipeline writes:

- `reports/` (quality report + metrics JSON)

- `figures/` (diagnostic plots + chain panels + consensus panels)

- `artifacts/` (consensus crowns GeoJSON/GPKG)


In [2]:
from __future__ import annotations



from pathlib import Path

import sys





def find_repo_root(start: Path) -> Path:

    start = start.resolve()

    for cand in [start, *start.parents]:

        if (cand / "src").is_dir() and (cand / "input").is_dir() and (cand / "output").is_dir():

            return cand

    raise RuntimeError(

        "Could not find repo root (expected folders: src/, input/, output/) starting from "

        f"{start}"

    )





REPO_ROOT = find_repo_root(Path.cwd())

SRC_DIR = REPO_ROOT / "src"

if str(SRC_DIR) not in sys.path:

    sys.path.insert(0, str(SRC_DIR))



print("CWD:", Path.cwd())

print("Repo root:", REPO_ROOT)

print("Using src:", SRC_DIR)


CWD: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/src/organised
Repo root: /Users/hbot07/VS Code/Drone-Phenology-Monitoring
Using src: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/src


In [3]:
from organised.tracking import TrackingConfig, run_tracking_pipeline



# --- SIT inputs ---

SIT_ORTHOS_DIR = REPO_ROOT / "input" / "input_om_sit"

SIT_CROWNS_DIR = REPO_ROOT / "output" / "input_crowns_multithreshold_min0p15"



# --- Run output root (everything saved under here) ---

RUN_OUTPUT_DIR = REPO_ROOT / "output" / "organised_sit_run"



print("SIT orthos dir:", SIT_ORTHOS_DIR)

print("SIT crowns dir:", SIT_CROWNS_DIR)

print("Run output dir:", RUN_OUTPUT_DIR)



assert SIT_ORTHOS_DIR.exists(), f"Missing: {SIT_ORTHOS_DIR}"

assert SIT_CROWNS_DIR.exists(), f"Missing: {SIT_CROWNS_DIR}"

RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


SIT orthos dir: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/input/input_om_sit
SIT crowns dir: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/input_crowns_multithreshold_min0p15
Run output dir: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/organised_sit_run


In [5]:
cfg = TrackingConfig(

    ortho_dir=SIT_ORTHOS_DIR,

    multithresh_dir=SIT_CROWNS_DIR,

    output_dir=RUN_OUTPUT_DIR,



    # Alignment

    use_phase_corr_alignment=True,



    # Graph build / matching

    classify_mode="strict",  # strict|balanced|lite



    # Chain selection for consensus

    include_partial_for_consensus=True,

    min_partial_len=5,

    min_partial_one_to_one_ratio=0.9,



    # Gap-fill augmentation

    enable_gap_fill=True,

    gapfill_min_threshold_tag="conf_0p15",

)



summary, artifacts, report_md = run_tracking_pipeline(cfg)


In [6]:
print(report_md)



print("\n--- Key outputs ---")

print("Consensus crowns:", summary.n_consensus)

print("OMs:", summary.n_oms)

print("Nodes:", summary.n_nodes, "Edges:", summary.n_edges)

print("Total extracted chains:", summary.n_chains_total)



print("\n--- Written to disk ---")

print("Output dir:", artifacts.output_dir)

print("Report:", artifacts.quality_report_path)

print("Metrics:", artifacts.metrics_json_path)

print("Consensus (GeoJSON):", artifacts.consensus_geojson_path)

print("Consensus (GPKG):", artifacts.consensus_gpkg_path)

print("Chain panels:", artifacts.chain_panels_dir)

print("Consensus panels:", artifacts.consensus_panels_dir)


# Tree Tracking Quality Report
n_oms: 7
total_nodes: 1871
total_edges: 1455
overall_match_rate: 1.111
average_chain_length: 1.91
median_chain_length: 1.00
max_chain_length: 7

## Match Rates by Consecutive OM Pair
- 1->2: 167/145 (1.152)
- 2->3: 295/293 (1.007)
- 3->4: 281/277 (1.014)
- 4->5: 304/277 (1.097)
- 5->6: 173/159 (1.088)
- 6->7: 235/159 (1.478)

## Chain Length Distribution
- L=1: 742
- L=2: 82
- L=3: 83
- L=4: 38
- L=5: 53
- L=6: 12
- L=7: 57

## Edge Selection by Case
- containment: selected 47 / candidates 283 (ratio 0.17)
- nearby: selected 1129 / candidates 2327 (ratio 0.49)
- one_to_one: selected 202 / candidates 206 (ratio 0.98)

## Graph Complexity
- avg_in_degree: 0.78
- avg_out_degree: 0.78
- branching_nodes: 674
- max_in_degree: 2
- max_out_degree: 2

--- Key outputs ---
Consensus crowns: 63
OMs: 7
Nodes: 1871 Edges: 1455
Total extracted chains: 1067

--- Written to disk ---
Output dir: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/organised_sit_run
Repo

In [7]:
# --- Quick tuning run (panels disabled for speed) ---

TUNED_OUTPUT_DIR = REPO_ROOT / "output" / "organised_sit_run_tuned_min3_ratio0p8"

TUNED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)



cfg_tuned = TrackingConfig(

    ortho_dir=SIT_ORTHOS_DIR,

    multithresh_dir=SIT_CROWNS_DIR,

    output_dir=TUNED_OUTPUT_DIR,

    use_phase_corr_alignment=True,

    classify_mode="strict",

    include_partial_for_consensus=True,

    min_partial_len=3,

    min_partial_one_to_one_ratio=0.8,

    enable_gap_fill=True,

    gapfill_min_threshold_tag="conf_0p15",

    save_chain_panels=False,

    save_consensus_panels=False,

)



summary_tuned, artifacts_tuned, report_md_tuned = run_tracking_pipeline(cfg_tuned)

print("Tuned consensus crowns:", summary_tuned.n_consensus)

print("Tuned output dir:", artifacts_tuned.output_dir)


Tuned consensus crowns: 76
Tuned output dir: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/organised_sit_run_tuned_min3_ratio0p8


In [8]:
# --- Full tuned run with figures/panels ---

# This can be slow because panels require raster patch extraction.

FULL_OUTPUT_DIR = REPO_ROOT / "output" / "organised_sit_run_tuned_with_panels"

FULL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)



cfg_full = TrackingConfig(

    ortho_dir=SIT_ORTHOS_DIR,

    multithresh_dir=SIT_CROWNS_DIR,

    output_dir=FULL_OUTPUT_DIR,

    use_phase_corr_alignment=True,

    classify_mode="strict",

    include_partial_for_consensus=True,

    min_partial_len=3,

    min_partial_one_to_one_ratio=0.8,

    enable_gap_fill=True,

    gapfill_min_threshold_tag="conf_0p15",

    save_chain_panels=True,

    save_consensus_panels=True,

    max_panels=120,

)



summary_full, artifacts_full, report_md_full = run_tracking_pipeline(cfg_full)

print("Full tuned consensus crowns:", summary_full.n_consensus)

print("Full tuned output dir:", artifacts_full.output_dir)


Full tuned consensus crowns: 76
Full tuned output dir: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/organised_sit_run_tuned_with_panels
